In [ ]:
# Machine-specific paths live in config.yaml next to this notebook (gitignored).
# Copy config.example.yaml -> config.yaml and fill in your paths.
from phoreal.config import load_config
cfg = load_config('config.yaml')

### Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

#Import general libs you'll need
import os
import geopandas as gpd
from shapely.geometry import Point

# Import functions from main
from phoreal.io import readers, processing
from phoreal import analysis
from phoreal.agg_configs import ATL_AGG_CONFIG, ATL08_AGG_CONFIG

# Import functions from the utils
from phoreal import plotting as plotter

### Identify files



In [ ]:
atl03_dir = cfg['florida_atl03_dir']
atl03_name = 'ATL03_20200212094736_07280607_006_01.h5'
atl03_file = os.path.join(atl03_dir, atl03_name)

# Define ATL08 File
atl08_dir = cfg['florida_atl08_dir']
atl08_name = 'ATL08_20200212094736_07280607_006_01.h5'
atl08_file = os.path.join(atl08_dir, atl08_name)

# Define ATL24 file
atl24_dir = cfg['florida_atl24_dir']
atl24_name = 'ATL24_20200212094736_07280607_006_01_001_01.h5'
atl24_file = os.path.join(atl24_dir, atl24_name)

# Define the ground track of interest
gt = 'gt1r'

### Read ATL03 File (photon rate data)


In [ ]:
# Get photon rate DF
df_ph = readers.read_photon_dataframe(atl03_file, gt, atl08_file)

### Read ATL08 File (Land segment level data)

In [ ]:
df_atl08 = readers.read_atl08(atl08_file, gt, atl03_file = atl03_file)

In [ ]:
df_atl08.columns

### Read ATL08 (20 m segment)

We can read in an ATL08 file to do this.

In [ ]:
df_atl08_20m = readers.read_atl08_20m(atl08_file, gt)

This can also be done by "expanding" an already read in ATL08

In [ ]:
df_atl08_20m = readers.expand_atl08_20m_df(df_atl08)

In [ ]:
import numpy as np
print(len(np.unique(df_atl08_20m['alongtrack'])))
print(len(df_atl08_20m))

### Reaggregating ATL03 

In [ ]:
df_seg = analysis.aggregate_by_segment(df_ph, ATL08_AGG_CONFIG, res=30)

In [ ]:
df_seg.columns

In [ ]:
from phoreal.io.processing import add_atl08_metadata_to_segments
df_seg = add_atl08_metadata_to_segments(df_seg, df_atl08)

In [ ]:
df_seg['photon_rate_can'] = (df_seg['n_ca_photons'] + df_seg['n_toc_photons']) / df_seg['n_shots']
df_seg['photon_rate_te'] = df_seg['n_te_photons'] / df_seg['n_shots']

In [ ]:
# Write out as geopandas dataframe
geometry_seg = [Point(xy) for xy in zip(df_seg.longitude, df_seg.latitude)]
gdf_seg = gpd.GeoDataFrame(df_seg, geometry=geometry_seg, crs="EPSG:4326")

# Save geopandas dataframe
gdf_seg.to_file("is2_topobathy.gpkg", layer='atl08atl24', driver="GPKG")